# Power, minimum detectable effect, sample size

Every number here comes from one normal-model formula. A two-arm difference in means with
`n` units in total, outcome sd `sd` and treated share `allocation` has standard error
`se = sd / sqrt(n · p · (1 − p))`; the power to detect `δ` at level `α` is then

    two-sided:  Φ(|δ|/se − z_{1−α/2}) + Φ(−|δ|/se − z_{1−α/2})

`mde` and `sample_size` are *exact* inversions of that formula (not the textbook
`(z_α + z_β)·se` approximation), so `power(mde(...))` reproduces the target. Every result is
a `Spec` that carries `alpha`, `power` and `two_sided` with it.

In [ ]:
import numpy as np

from axiom.core import D, Posterior, Unit, Unsupported
from axiom.design import (
    MDE, AnchoredEffect, Assignment, ClusterDesign, CostPerOutcomeInterval, CostPerOutcomePower,
    HoldoutTradeoff, MatchMetric, PowerCurve, PowerResult, SampleSize, anchor_draws, anchor_effect,
    cluster_mde, cluster_power, clusters_needed, coefficient_mde, coefficient_power,
    coefficient_sample_size, cost_per_outcome_interval, cost_per_outcome_power, design_effect,
    difference_se, effective_sample_size, holdout_tradeoff, match_clusters,
    max_detectable_cost_per_outcome, mde, power, power_curve, power_from_se, sample_size,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

In [ ]:
se = difference_se(200, sd=2.0, allocation=0.5)
pr: PowerResult = power(200, effect=0.6, sd=2.0)
print(f"se = {se:.4f}  power = {pr.power:.3f}  (alpha={pr.alpha}, two_sided={pr.two_sided})")
print("one-sided, 40% treated:", round(power(200, 0.6, 2.0, two_sided=False, allocation=0.4).power, 3))

In [ ]:
m: MDE = mde(200, sd=2.0, power=0.8)
print(f"MDE at n=200: {m.effect:.4f}   check power(mde) = {power(200, m.effect, 2.0).power:.6f}")
ss = sample_size(effect=0.6, sd=2.0, power=0.8)
assert isinstance(ss, SampleSize)
print(f"n for effect 0.6: {ss.n} ({ss.n_treated} treated / {ss.n_control} control), achieved power {ss.power:.4f}")
print("zero effect ->", type(sample_size(0.0, 2.0)).__name__)

In [ ]:
curve: PowerCurve = power_curve(200, 2.0, effects=np.linspace(0.0, 1.2, 7))
table(
    [[f"{e:.2f}", f"{p:.3f}"] for e, p in zip(curve.effects, curve.powers)],
    headers=("effect", "power"),
)
print("interpolated at 0.5:", round(curve.power_at(0.5), 3))

## Regression coefficients

A coefficient has whatever design standard error the design gives it. The `coefficient_*`
functions take that `se` directly; for sample size they assume the design is replicated, so
`se(m) = se · sqrt(n / m)`.

In [ ]:
print("power_from_se:", round(power_from_se(0.3, 0.1).power, 3))
print("coefficient_power:", round(coefficient_power(0.3, se=0.1).power, 3))
print("coefficient_mde:", round(coefficient_mde(se=0.1, power=0.9).effect, 4))
css = coefficient_sample_size(effect=0.2, se=0.1, n=500)
assert isinstance(css, SampleSize)
print(f"coefficient_sample_size: n={css.n}, se at n={css.se:.4f}, power={css.power:.4f}")

## Cluster-randomized designs

Randomizing `k` clusters of `m` individuals is worth fewer than `k·m` independent observations
when outcomes within a cluster are correlated. With intra-cluster correlation `icc` the
inflation is the design effect `DE = 1 + (m − 1)·icc`; every cluster function reduces to
`power` with `n = k·m` and `sd·sqrt(DE)`. There is no second power formula.

In [ ]:
region = Unit(name="region", dimension=D.entity, kind="cluster")
cd = ClusterDesign(unit=region, n_clusters=24, cluster_size=40, icc=0.05, allocation=0.5)
print("design effect:", design_effect(40, 0.05), "| effective n:", round(effective_sample_size(24, 40, 0.05), 1))
print("cluster power for effect 0.5:", round(cluster_power(cd, effect=0.5, sd=2.0).power, 3))
print("cluster MDE:", round(cluster_mde(cd, sd=2.0).effect, 4))
cn = clusters_needed(effect=0.5, sd=2.0, cluster_size=40, icc=0.05)
assert isinstance(cn, SampleSize)
print(f"clusters needed: {cn.n} ({cn.n_treated} treated), achieved power {cn.power:.3f}")

In [ ]:
rng = np.random.default_rng(0)
pre = rng.normal(10.0, 2.0, size=(9, 1)) + rng.normal(0.0, 0.3, size=(9, 6))
metric: MatchMetric = "trajectory"
asg: Assignment = match_clusters(pre, labels=[f"r{i}" for i in range(9)], unit=region, metric=metric, seed=1)
print("pairs:", asg.pairs, "| unpaired:", asg.unpaired)
print("treated:", asg.treated, "control:", asg.control)
print(f"pre-period SMD {asg.pre_smd:.3f}, mean pair distance {asg.mean_pair_distance:.3f}")

In [ ]:
ht: HoldoutTradeoff = holdout_tradeoff(cd, sd=2.0, fractions=(0.1, 0.2, 0.3, 0.4, 0.5))
table(
    [[f"{f:.1f}", f"{m_:.3f}", f"{r:.2f}x"] for f, m_, r in zip(ht.fractions, ht.mdes, ht.relative_mde)],
    headers=("holdout", "MDE", "against the best"),
)
print("best fraction:", ht.best_fraction)

## Anchoring the MDE to a posterior

If a fitted model already puts most of its mass above the conventional MDE, the experiment is
powered to confirm a belief, not to test it. `anchor_effect` reports the posterior probability
of exceeding the MDE and the `1 − credence` quantile — the effect the model *doubts*, which is
what a test should be powered for.

In [ ]:
draws = rng.normal(0.9, 0.25, size=(2, 500))
posterior = Posterior({"beta": draws})
ae = anchor_effect(posterior, "beta", mde=0.5, credence=0.9)
assert isinstance(ae, AnchoredEffect)
print(f"P(beta > 0.5) = {ae.probability_exceeds_mde:.3f} (gaussian {ae.probability_exceeds_mde_gaussian:.3f})")
print(f"anchored effect (10% quantile) = {ae.anchored_effect:.3f}; already believed: {ae.already_believed}")
print("from flat draws:", round(anchor_draws(draws.ravel(), "beta", 0.5).anchored_effect, 3))
print("missing parameter ->", anchor_effect(posterior, "gamma", 0.5).reason)

## Cost per outcome unit

`cost / effect` has a noisy denominator, so its interval is asymmetric. The Fieller interval
inverts the effect's Wald interval, `cost / (effect ± z·se)`; when that interval reaches zero
the ratio is unbounded above and the result says so rather than printing a finite number. The
naive delta-method interval is reported alongside for comparison.

In [ ]:
ci: CostPerOutcomeInterval = cost_per_outcome_interval(cost=1000.0, effect=25.0, effect_se=6.0)
print(f"estimate {ci.estimate:.2f}  fieller [{ci.lower:.2f}, {ci.upper:.2f}]  status={ci.status}")
print("naive:", ci.naive_interval)
wide = cost_per_outcome_interval(1000.0, 25.0, 15.0)
print("noisy effect ->", wide.status, "upper =", wide.upper)
cp: CostPerOutcomePower = cost_per_outcome_power(1000.0, true_effect=25.0, effect_se=6.0, threshold=60.0)
print(f"P(upper bound < 60) = {cp.power:.3f}  (effect required {cp.effect_required:.2f})")
print("largest bounded cost per outcome at the MDE:", round(max_detectable_cost_per_outcome(1000.0, m.effect), 2))